# TS-ResNet50 120k Pilot Benchmark

This notebook brings the teacher-student ResNet50 idea from the stronger `CT` branch onto the labeled `120k / 10k / 20k` split with one controlled test.

Plan for this notebook:
- reuse the existing labeled `120k / 10k / 20k` metadata and array pipeline
- train one teacher-student model with a `ResNet50` teacher on `layer2`
- keep the architecture close to the friend-branch recipe: hidden dim `128`, `topk_mean`, and normal-manifold learning
- run a post-training score sweep over student and autoencoder branch weights plus wafer-level reduction rules
- compare the default TS score and the best rescored TS variant against local PatchCore controls when those artifacts are available

Not in scope for this first pass:
- teacher-layer ablations
- image-size sweeps
- ensemble experiments
- Modal packaging


In [1]:
from pathlib import Path
import json
import subprocess
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation.reconstruction_metrics import summarize_threshold_metrics, sweep_threshold_metrics
from wafer_defect.models.ts_distillation import build_ts_distillation_from_config
from wafer_defect.scoring import spatial_max, spatial_mean, topk_spatial_mean


In [2]:
CONFIG_PATH = REPO_ROOT / "configs" / "training" / "train_ts_resnet50_120k.toml"
DATA_CONFIG_PATH = REPO_ROOT / "configs" / "data" / "data_patchcore_wrn50_120k.toml"
PATCHCORE_BASELINE_SUMMARY = REPO_ROOT / "outputs" / "modal_runs" / "patchcore_wrn50_120k_20260322_validation_f1" / "patchcore_wrn50_multilayer_120k_5pct" / "bundle_summary.json"
PATCHCORE_NOTEBOOK4_RESULTS = REPO_ROOT / "outputs" / "modal_runs" / "patchcore_wrn50_120k_20260322_notebook4_fetch" / "patchcore_wrn50_multilayer_120k_notebook4" / "notebook4_results.csv"

THRESHOLD_QUANTILE = 0.95
RUN_DATA_PREP = False
RUN_TRAINING = True
RUN_SCORE_SWEEP = True

SCORE_SWEEP_WEIGHTS = [
    (1.0, 1.0),
    (1.0, 0.0),
    (0.0, 1.0),
    (2.0, 1.0),
    (1.0, 2.0),
    (1.0, 0.5),
    (0.5, 1.0),
]
SCORE_SWEEP_REDUCTIONS = [
    ("mean", None),
    ("max", None),
    ("topk_mean", 0.01),
    ("topk_mean", 0.05),
    ("topk_mean", 0.10),
    ("topk_mean", 0.20),
]

config = load_toml(CONFIG_PATH)
metadata_path = REPO_ROOT / config["data"]["metadata_csv"]
output_dir = REPO_ROOT / config["run"]["output_dir"]
evaluation_dir = output_dir / "evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)
best_model_path = output_dir / "best_model.pt"

display(config)


{'run': {'output_dir': 'artifacts/x64/ts_resnet50_120k', 'seed': 42},
 'data': {'metadata_csv': 'data/processed/x64/wm811k_patchcore_custom/metadata_train120000_a6000_val10000_a500_test20000_a1000.csv',
  'image_size': 64,
  'batch_size': 16,
  'num_workers': 0},
 'training': {'epochs': 30,
  'learning_rate': 0.0003,
  'weight_decay': 1e-05,
  'device': 'auto',
  'early_stopping_patience': 5,
  'early_stopping_min_delta': 0.0001,
  'checkpoint_every': 5,
  'resume_from': ''},
 'model': {'type': 'ts_distillation',
  'teacher_backbone': 'resnet50',
  'teacher_layer': 'layer2',
  'teacher_pretrained': True,
  'teacher_input_size': 224,
  'normalize_teacher_input': True,
  'feature_autoencoder_hidden_dim': 128,
  'student_weight': 1.0,
  'autoencoder_weight': 1.0,
  'score_student_weight': 1.0,
  'score_autoencoder_weight': 0.0,
  'reduction': 'topk_mean',
  'topk_ratio': 0.2}}

In [ ]:
def run_command(args, cwd=REPO_ROOT):
    command = [str(arg) for arg in args]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)


if RUN_DATA_PREP or not metadata_path.exists():
    run_command([
        sys.executable,
        "-u",
        REPO_ROOT / "scripts" / "prepare_wm811k.py",
        "--config",
        DATA_CONFIG_PATH,
    ])
else:
    print(f"Using existing metadata: {metadata_path}")

metadata_df = pd.read_csv(metadata_path)
split_summary = (
    metadata_df.groupby(["split", "is_anomaly"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["split", "is_anomaly"])
)
display(split_summary)


Running: c:\Users\genso\Documents\College_Projects\DeepLearning 2610\Project\.venv\Scripts\python.exe -u C:\Users\genso\Documents\College_Projects\DeepLearning 2610\Project\scripts\prepare_wm811k.py --config C:\Users\genso\Documents\College_Projects\DeepLearning 2610\Project\configs\data\data_patchcore_wrn50_120k.toml


In [ ]:
requested_device = str(config["training"].get("device", "auto"))
if requested_device == "auto":
    resolved_device_name = "cuda" if torch.cuda.is_available() else "cpu"
else:
    resolved_device_name = requested_device

device = torch.device(resolved_device_name)

cuda_summary = {
    "config_device": requested_device,
    "resolved_device": resolved_device_name,
    "torch_cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
}
if torch.cuda.is_available():
    cuda_summary["cuda_device_name"] = torch.cuda.get_device_name(0)

display(pd.Series(cuda_summary))


In [ ]:
if RUN_TRAINING or not best_model_path.exists():
    run_command([
        sys.executable,
        "-u",
        REPO_ROOT / "scripts" / "train_ts_distillation.py",
        "--config",
        CONFIG_PATH,
    ])
else:
    print(f"Skipping training because checkpoint already exists: {best_model_path}")

history_path = output_dir / "history.json"
summary_path = output_dir / "summary.json"
history_df = pd.read_json(history_path)
training_summary = json.loads(summary_path.read_text(encoding="utf-8"))

display(pd.Series(training_summary))
display(history_df.tail())

if not history_df.empty:
    ax = history_df.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(8, 4), title="TS-ResNet50 120k loss")
    ax.set_ylabel("loss")
    plt.show()


## Score Sweep

The `CT` branch result improved after rescoring the same checkpoint, so this notebook keeps that idea intact by sweeping branch weights and wafer-level reductions on the saved `best_model.pt`.


In [ ]:
def collect_normalized_maps(model, dataloader, device):
    student_maps = []
    auto_maps = []
    labels = []
    model.eval()
    with torch.inference_mode():
        for inputs, batch_labels in dataloader:
            inputs = inputs.to(device)
            student_map, auto_map = model.raw_anomaly_maps(inputs)
            student_maps.append((student_map / model.student_map_scale.clamp_min(1e-6)).cpu())
            auto_maps.append((auto_map / model.autoencoder_map_scale.clamp_min(1e-6)).cpu())
            labels.append(batch_labels.cpu())
    return torch.cat(student_maps, dim=0), torch.cat(auto_maps, dim=0), torch.cat(labels, dim=0).numpy()


def reduce_anomaly_map(anomaly_map, reduction, topk_ratio):
    if reduction == "mean":
        return spatial_mean(anomaly_map)
    if reduction == "max":
        return spatial_max(anomaly_map)
    return topk_spatial_mean(anomaly_map, topk_ratio=topk_ratio)


def summarize_variant(name, student_weight, auto_weight, reduction, topk_ratio, val_scores, test_scores, val_labels, test_labels):
    val_normal_scores = val_scores[val_labels == 0]
    threshold = float(pd.Series(val_normal_scores).quantile(THRESHOLD_QUANTILE))
    metrics = summarize_threshold_metrics(test_labels, test_scores, threshold)
    _, best_sweep = sweep_threshold_metrics(test_labels, test_scores)
    return {
        "name": name,
        "student_weight": float(student_weight),
        "auto_weight": float(auto_weight),
        "reduction": reduction,
        "topk_ratio": None if topk_ratio is None else float(topk_ratio),
        "threshold": threshold,
        "precision": float(metrics["precision"]),
        "recall": float(metrics["recall"]),
        "f1": float(metrics["f1"]),
        "auroc": float(metrics["auroc"]),
        "auprc": float(metrics["auprc"]),
        "predicted_anomalies": int(metrics["predicted_anomalies"]),
        "best_sweep_threshold": float(best_sweep["threshold"]),
        "best_sweep_precision": float(best_sweep["precision"]),
        "best_sweep_recall": float(best_sweep["recall"]),
        "best_sweep_f1": float(best_sweep["f1"]),
        "best_sweep_predicted_anomalies": int(best_sweep["predicted_anomalies"]),
    }


In [ ]:
image_size = int(config["data"].get("image_size", 64))
batch_size = int(config["data"].get("batch_size", 16))
num_workers = int(config["data"].get("num_workers", 0))

val_dataset = WaferMapDataset(config["data"]["metadata_csv"], split="val", image_size=image_size)
test_dataset = WaferMapDataset(config["data"]["metadata_csv"], split="test", image_size=image_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

checkpoint = torch.load(best_model_path, map_location="cpu")
score_sweep_model = build_ts_distillation_from_config(config, image_size=image_size)
score_sweep_model.load_state_dict(checkpoint["model_state_dict"])
score_sweep_model = score_sweep_model.to(device)
score_sweep_model.eval()

val_student_maps, val_auto_maps, val_labels = collect_normalized_maps(score_sweep_model, val_loader, device)
test_student_maps, test_auto_maps, test_labels = collect_normalized_maps(score_sweep_model, test_loader, device)

default_student_weight = float(config["model"].get("score_student_weight", config["model"].get("student_weight", 1.0)))
default_auto_weight = float(config["model"].get("score_autoencoder_weight", config["model"].get("autoencoder_weight", 1.0)))
default_reduction = str(config["model"].get("reduction", "topk_mean"))
default_topk_ratio = float(config["model"].get("topk_ratio", 0.20))

default_val_map = default_student_weight * val_student_maps + default_auto_weight * val_auto_maps
default_test_map = default_student_weight * test_student_maps + default_auto_weight * test_auto_maps
default_val_scores = reduce_anomaly_map(default_val_map, default_reduction, default_topk_ratio).numpy()
default_test_scores = reduce_anomaly_map(default_test_map, default_reduction, default_topk_ratio).numpy()

score_rows = [
    summarize_variant(
        name="default_config",
        student_weight=default_student_weight,
        auto_weight=default_auto_weight,
        reduction=default_reduction,
        topk_ratio=default_topk_ratio,
        val_scores=default_val_scores,
        test_scores=default_test_scores,
        val_labels=val_labels,
        test_labels=test_labels,
    )
]

if RUN_SCORE_SWEEP:
    for student_weight, auto_weight in SCORE_SWEEP_WEIGHTS:
        val_map = student_weight * val_student_maps + auto_weight * val_auto_maps
        test_map = student_weight * test_student_maps + auto_weight * test_auto_maps
        for reduction, topk_ratio in SCORE_SWEEP_REDUCTIONS:
            val_scores = reduce_anomaly_map(val_map, reduction, topk_ratio).numpy()
            test_scores = reduce_anomaly_map(test_map, reduction, topk_ratio).numpy()
            score_rows.append(
                summarize_variant(
                    name=f"s{student_weight:g}_a{auto_weight:g}_{reduction}" + ("" if topk_ratio is None else f"_r{topk_ratio:.2f}"),
                    student_weight=student_weight,
                    auto_weight=auto_weight,
                    reduction=reduction,
                    topk_ratio=topk_ratio,
                    val_scores=val_scores,
                    test_scores=test_scores,
                    val_labels=val_labels,
                    test_labels=test_labels,
                )
            )

score_sweep_df = pd.DataFrame(score_rows).sort_values(["f1", "auprc", "auroc"], ascending=False).reset_index(drop=True)
score_sweep_df.to_csv(evaluation_dir / "score_sweep_summary.csv", index=False)
selected_variant = json.loads(score_sweep_df.iloc[0].to_json())
(evaluation_dir / "selected_score_variant.json").write_text(json.dumps(selected_variant, indent=2), encoding="utf-8")

display(score_sweep_df.head(10).round(6))
display(pd.Series(selected_variant))


In [ ]:
comparison_rows = [
    {
        "model": "TS-Res50 120k best",
        "name": selected_variant["name"],
        "precision": selected_variant["precision"],
        "recall": selected_variant["recall"],
        "f1": selected_variant["f1"],
        "auroc": selected_variant["auroc"],
        "auprc": selected_variant["auprc"],
        "predicted_anomalies": selected_variant["predicted_anomalies"],
    },
    {
        "model": "TS-Res50 120k default",
        "name": score_rows[0]["name"],
        "precision": score_rows[0]["precision"],
        "recall": score_rows[0]["recall"],
        "f1": score_rows[0]["f1"],
        "auroc": score_rows[0]["auroc"],
        "auprc": score_rows[0]["auprc"],
        "predicted_anomalies": score_rows[0]["predicted_anomalies"],
    },
]

if PATCHCORE_BASELINE_SUMMARY.exists():
    patchcore_summary = json.loads(PATCHCORE_BASELINE_SUMMARY.read_text(encoding="utf-8"))
    selected_name = patchcore_summary["selected_variant"]
    patchcore_row = next(row for row in patchcore_summary["variants"] if row["name"] == selected_name)
    comparison_rows.append(
        {
            "model": "PatchCore WRN50 baseline",
            "name": patchcore_row["name"],
            "precision": patchcore_row["precision"],
            "recall": patchcore_row["recall"],
            "f1": patchcore_row["f1"],
            "auroc": patchcore_row["auroc"],
            "auprc": patchcore_row["auprc"],
            "predicted_anomalies": patchcore_row["predicted_anomalies"],
        }
    )

if PATCHCORE_NOTEBOOK4_RESULTS.exists():
    patchcore_notebook4 = pd.read_csv(PATCHCORE_NOTEBOOK4_RESULTS)
    for variant_name, label in [
        ("coverage__64_l23_normals_2048src", "PatchCore WRN50 2048src"),
        ("coverage__64_l23_normals_8192src", "PatchCore WRN50 8192src"),
    ]:
        matched = patchcore_notebook4.loc[patchcore_notebook4["name"] == variant_name]
        if matched.empty:
            continue
        row = matched.iloc[0]
        comparison_rows.append(
            {
                "model": label,
                "name": row["name"],
                "precision": float(row["precision"]),
                "recall": float(row["recall"]),
                "f1": float(row["f1"]),
                "auroc": float(row["auroc"]),
                "auprc": float(row["auprc"]),
                "predicted_anomalies": int(row["predicted_anomalies"]),
            }
        )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(evaluation_dir / "patchcore_reference_comparison.csv", index=False)
display(comparison_df.round(6))
